
## Confirmatory Factor Analysis (CFA) of PHQ-9
Two models are compared for Self-Report, GPT-4, and GPT-5:
- **1-factor (unidimensional):** all 9 items load on a single *Depression* latent factor
- **2-factor (correlated):** *Somatic* (sleep, fatigue, appetite, psychomotor) vs. *Cognitive/Affective* (anhedonia, depressed mood, worthlessness, concentration, suicidal)

Estimated via ML using `semopy`. Model fit compared with CFI, TLI, RMSEA, SRMR, AIC, BIC, and a chi-square difference test.

In [1]:
import pandas as pd
import numpy as np
# pip install semopy  (if not already installed)
import semopy
from scipy.stats import chi2 as chi2_dist
from IPython.display import display


In [2]:
# ── Load data ─────────────────────────────────────────────────────────────────
# Adjust paths to match your environment
expert1_file = '/cronus_data/avirinchipur/reasoning_for_psych/expts/parsed_responses/Item_level_scores_Katarina.csv'
expert2_file = '/cronus_data/avirinchipur/reasoning_for_psych/expts/parsed_responses/Item_level_scores_Veerle.csv'
self_report_file = '/cronus_data/avirinchipur/reasoning_for_psych/expts/parsed_responses/self_report_unified.csv'
gpt4_file = '/cronus_data/avirinchipur/reasoning_for_psych/expts/parsed_responses/expt_gpt-4-1106-preview.dep_list_phq9items_score_classify2_editted_unified.csv'
gpt5_file = '/chronos_data/avirinchipur/reasoning_for_psych/expts/parsed_responses/expt_gpt-5.dep_list_phq9items_score_classify2.csv'

self_report_df = pd.read_csv(self_report_file, index_col=0)
gpt4_df        = pd.read_csv(gpt4_file,        index_col=0)
gpt5_df        = pd.read_csv(gpt5_file,        index_col=0)

print('Self-Report:', self_report_df.shape)
print('GPT-4:      ', gpt4_df.shape)
print('GPT-5:      ', gpt5_df.shape)


Self-Report: (963, 10)
GPT-4:       (955, 28)
GPT-5:       (956, 28)


In [3]:
phq9_items = [
    'score_Anhedonia', 'score_Depressed_Mood', 'score_Insomnia_or_Hypersomnia',
    'score_Fatigue', 'score_Poor_appetite_or_overeating', 'score_Worthlessness_or_Guilt',
    'score_Difficulty_concentrating', 'score_Psychomotor_agitation_or_retardation',
    'score_Suicidal_ideation'
]

# Model 1 — Unidimensional
model_1factor = """
    Depression =~ score_Anhedonia + score_Depressed_Mood + score_Insomnia_or_Hypersomnia + score_Fatigue + score_Poor_appetite_or_overeating + score_Worthlessness_or_Guilt + score_Difficulty_concentrating + score_Psychomotor_agitation_or_retardation + score_Suicidal_ideation
"""

# Model 2 — Two-factor (Somatic vs. Cognitive/Affective), factors allowed to correlate
model_2factor = """
    Somatic   =~ score_Insomnia_or_Hypersomnia + score_Fatigue + score_Poor_appetite_or_overeating + score_Psychomotor_agitation_or_retardation
    Cognitive =~ score_Anhedonia + score_Depressed_Mood + score_Worthlessness_or_Guilt + score_Difficulty_concentrating + score_Suicidal_ideation
"""


In [4]:
raters = {
    'Self-Report': self_report_df,
    'GPT-4':       gpt4_df,
    'GPT-5':       gpt5_df,
}

# Keep only rows present in all three raters
common_idx = (self_report_df.index
              .intersection(gpt4_df.index)
              .intersection(gpt5_df.index))
print(f'Common samples across all raters: {len(common_idx)}')

cfa_results = {}
for rater_name, df in raters.items():
    cfa_results[rater_name] = {}
    data = df.loc[common_idx, phq9_items].dropna()
    print(f'\nFitting CFA for {rater_name} (N={len(data)})...')
    for model_name, model_desc in [('1-factor', model_1factor), ('2-factor', model_2factor)]:
        m = semopy.Model(model_desc)
        m.fit(data)
        stats = semopy.calc_stats(m)
        cfa_results[rater_name][model_name] = {'model': m, 'stats': stats}
        try:
            cfi   = float(stats.loc['Value', 'CFI'])
            rmsea = float(stats.loc['Value', 'RMSEA'])
            aic   = float(stats.loc['Value', 'AIC'])
            print(f'  {model_name}: CFI={cfi:.4f}, RMSEA={rmsea:.4f}, AIC={aic:.2f}')
        except Exception:
            print(f'  {model_name} stats:\n', stats)


Common samples across all raters: 955

Fitting CFA for Self-Report (N=955)...
  1-factor: CFI=0.9253, RMSEA=0.1271, AIC=35.07
  2-factor: CFI=0.9497, RMSEA=0.1063, AIC=37.36

Fitting CFA for GPT-4 (N=955)...
  1-factor: CFI=0.9160, RMSEA=0.1383, AIC=34.91
  2-factor: CFI=0.9268, RMSEA=0.1315, AIC=37.05

Fitting CFA for GPT-5 (N=955)...
  1-factor: CFI=0.8500, RMSEA=0.1362, AIC=34.94
  2-factor: CFI=0.9134, RMSEA=0.1054, AIC=37.37


In [5]:
# Fit-index summary table
# Thresholds: CFI/TLI > 0.95 good | RMSEA < 0.06 (good) / < 0.08 (acceptable) | SRMR < 0.08 good

stat_keys = [
    ('chi2',    'chi2'),
    ('df',      'DoF'),
    ('p(chi2)', 'chi2 p-value'),
    ('CFI',     'CFI'),
    ('TLI',     'TLI'),
    ('RMSEA',   'RMSEA'),
    ('AIC',     'AIC'),
    ('BIC',     'BIC'),
]

rows = []
for rater_name in raters:
    for model_name in ['1-factor', '2-factor']:
        stats = cfa_results[rater_name][model_name]['stats']
        row = {'Rater': rater_name, 'Model': model_name}
        for col_name, key in stat_keys:
            try:
                row[col_name] = round(float(stats.loc['Value', key]), 4)
            except KeyError:
                row[col_name] = None
        rows.append(row)

summary_df = pd.DataFrame(rows).set_index(['Rater', 'Model'])
print('Thresholds — CFI/TLI: >0.95 | RMSEA: <0.06 (good) / <0.08 (acceptable) | SRMR: <0.08')
display(summary_df)


Thresholds — CFI/TLI: >0.95 | RMSEA: <0.06 (good) / <0.08 (acceptable) | SRMR: <0.08


chi2    df p(chi2)   CFI   TLI RMSEA   AIC   BIC
Rater       Model                                                     
Self-Report 1-factor  None  None    None  None  None  None  None  None
            2-factor  None  None    None  None  None  None  None  None
GPT-4       1-factor  None  None    None  None  None  None  None  None
            2-factor  None  None    None  None  None  None  None  None
GPT-5       1-factor  None  None    None  None  None  None  None  None
            2-factor  None  None    None  None  None  None  None  None

In [8]:
# Standardized factor loadings
print('=' * 70)
print('Standardized Factor Loadings')
print('=' * 70)

for rater_name in raters:
    for model_name in ['1-factor', '2-factor']:
        m = cfa_results[rater_name][model_name]['model']
        print(f'\n--- {rater_name} | {model_name} ---')
        insp = m.inspect(std_est=True)
        loadings = insp[insp['op'] == '~'][['lval', 'rval', 'Estimate', 'Std. Err', 'z-value', 'p-value', 'Est. Std']]
        loadings = loadings.rename(columns={'lval': 'Factor', 'rval': 'Item'})
        display(loadings.reset_index(drop=True))


Standardized Factor Loadings

--- Self-Report | 1-factor ---


,Factor,Item,Estimate,Std. Err,z-value,p-value,Est. Std
0,score_Anhedonia,Depression,1.000000,-,-,-,0.831603
1,score_Depressed_Mood,Depression,1.086616,0.032066,33.88664,0.0,0.876540
2,score_Insomnia_or_Hypersomnia,Depression,1.012347,0.037309,27.13447,0.0,0.757641
3,score_Fatigue,Depression,1.016249,0.03367,30.182651,0.0,0.814288
4,score_Poor_appetite_or_overeating,Depression,0.964303,0.037818,25.498723,0.0,0.724878
5,score_Worthlessness_or_Guilt,Depression,1.035602,0.035268,29.363618,0.0,0.799603
6,score_Difficulty_concentrating,Depression,0.930338,0.033575,27.709094,0.0,0.768747
7,score_Psychomotor_agitation_or_retardation,Depression,0.616400,0.032978,18.691267,0.0,0.569200
8,score_Suicidal_ideation,Depression,0.692804,0.032783,21.133239,0.0,0.628717



--- Self-Report | 2-factor ---


,Factor,Item,Estimate,Std. Err,z-value,p-value,Est. Std
0,score_Insomnia_or_Hypersomnia,Somatic,1.000000,-,-,-,0.817473
1,score_Fatigue,Somatic,0.983387,0.03225,30.492528,0.0,0.860679
2,score_Poor_appetite_or_overeating,Somatic,0.927858,0.03574,25.961598,0.0,0.761824
3,score_Psychomotor_agitation_or_retardation,Somatic,0.541190,0.031389,17.241366,0.0,0.545945
4,score_Anhedonia,Cognitive,1.000000,-,-,-,0.841268
5,score_Depressed_Mood,Cognitive,1.100945,0.030891,35.639154,0.0,0.898500
6,score_Worthlessness_or_Guilt,Cognitive,1.032905,0.034379,30.044666,0.0,0.806913
7,score_Difficulty_concentrating,Cognitive,0.911335,0.033072,27.556193,0.0,0.761866
8,score_Suicidal_ideation,Cognitive,0.700261,0.032084,21.826052,0.0,0.643067



--- GPT-4 | 1-factor ---


...

,Factor,Item,Estimate,Std. Err,z-value,p-value,Est. Std
0,score_Insomnia_or_Hypersomnia,Somatic,1.000000,-,-,-,0.838630
1,score_Fatigue,Somatic,1.118241,0.03463,32.291159,0.0,0.853288
2,score_Poor_appetite_or_overeating,Somatic,0.760235,0.029388,25.869042,0.0,0.734681
3,score_Psychomotor_agitation_or_retardation,Somatic,0.759998,0.027178,27.96402,0.0,0.775763
4,score_Anhedonia,Cognitive,1.000000,-,-,-,0.857410
5,score_Depressed_Mood,Cognitive,1.192886,0.033913,35.174459,0.0,0.873159
6,score_Worthlessness_or_Guilt,Cognitive,0.875924,0.031166,28.105493,0.0,0.761215
7,score_Difficulty_concentrating,Cognitive,0.658522,0.022649,29.07482,0.0,0.778088
8,score_Suicidal_ideation,Cognitive,0.301567,0.023516,12.823711,0.0,0.408530



--- GPT-5 | 1-factor ---


,Factor,Item,Estimate,Std. Err,z-value,p-value,Est. Std
0,score_Anhedonia,Depression,1.000000,-,-,-,0.817959
1,score_Depressed_Mood,Depression,1.200979,0.048969,24.525492,0.0,0.745386
2,score_Insomnia_or_Hypersomnia,Depression,0.790085,0.033207,23.792737,0.0,0.727327
3,score_Fatigue,Depression,0.928089,0.036128,25.689191,0.0,0.773876
4,score_Poor_appetite_or_overeating,Depression,0.344563,0.022811,15.105273,0.0,0.493163
5,score_Worthlessness_or_Guilt,Depression,0.520273,0.037609,13.833782,0.0,0.455315
6,score_Difficulty_concentrating,Depression,0.583801,0.027878,20.941098,0.0,0.654989
7,score_Psychomotor_agitation_or_retardation,Depression,0.251944,0.018074,13.939937,0.0,0.458509
8,score_Suicidal_ideation,Depression,0.251131,0.026817,9.364508,0.0,0.315607



--- GPT-5 | 2-factor ---


,Factor,Item,Estimate,Std. Err,z-value,p-value,Est. Std
0,score_Insomnia_or_Hypersomnia,Somatic,1.000000,-,-,-,0.799807
1,score_Fatigue,Somatic,1.168867,0.045915,25.457373,0.0,0.846840
2,score_Poor_appetite_or_overeating,Somatic,0.410651,0.026986,15.217078,0.0,0.510762
3,score_Psychomotor_agitation_or_retardation,Somatic,0.299558,0.021341,14.037046,0.0,0.473714
4,score_Anhedonia,Cognitive,1.000000,-,-,-,0.876867
5,score_Depressed_Mood,Cognitive,1.209672,0.043341,27.910565,0.0,0.804808
6,score_Worthlessness_or_Guilt,Cognitive,0.514374,0.034392,14.956264,0.0,0.482635
7,score_Difficulty_concentrating,Cognitive,0.511321,0.025661,19.925985,0.0,0.614957
8,score_Suicidal_ideation,Cognitive,0.253262,0.024785,10.218259,0.0,0.341229


In [13]:
# Comparison table: standardized loadings across raters — 1-factor model
item_labels = {
    'score_Anhedonia':                            'Anhedonia',
    'score_Depressed_Mood':                       'Depressed Mood',
    'score_Insomnia_or_Hypersomnia':              'Sleep',
    'score_Fatigue':                              'Fatigue',
    'score_Poor_appetite_or_overeating':          'Appetite',
    'score_Worthlessness_or_Guilt':               'Worthlessness / Guilt',
    'score_Difficulty_concentrating':             'Difficulty Concentrating',
    'score_Psychomotor_agitation_or_retardation': 'Psychomotor',
    'score_Suicidal_ideation':                    'Suicidal Ideation',
}

def get_loadings(rater_name, model_name):
    m = cfa_results[rater_name][model_name]['model']
    insp = m.inspect(std_est=True)
    loadings = insp[insp['op'] == '~'][['lval', 'Est. Std']].copy()  # lval = item
    loadings['Item'] = loadings['lval'].map(item_labels)
    return loadings.set_index('Item')['Est. Std']

rows_1f = {}
for rater_name in ['Self-Report', 'GPT-4', 'GPT-5']:
    rows_1f[rater_name] = get_loadings(rater_name, '1-factor')

compare_1f = pd.DataFrame(rows_1f).round(3)
compare_1f.index.name = 'PHQ-9 Item'
print('1-Factor Model — Standardized Loadings')
display(compare_1f)


1-Factor Model — Standardized Loadings


,Self-Report,GPT-4,GPT-5
PHQ-9 Item,,,
Anhedonia,0.832,0.841,0.818
Depressed Mood,0.877,0.847,0.745
Sleep,0.758,0.820,0.727
Fatigue,0.814,0.841,0.774
Appetite,0.725,0.721,0.493
Worthlessness / Guilt,0.800,0.746,0.455
Difficulty Concentrating,0.769,0.791,0.655
Psychomotor,0.569,0.763,0.459
Suicidal Ideation,0.629,0.396,0.316


In [14]:
# Comparison table: standardized loadings across raters — 2-factor model
def get_loadings_2f(rater_name):
    m = cfa_results[rater_name]['2-factor']['model']
    insp = m.inspect(std_est=True)
    loadings = insp[insp['op'] == '~'][['lval', 'rval', 'Est. Std']].copy()
    loadings['Item']   = loadings['lval'].map(item_labels)  # lval = item
    loadings['Factor'] = loadings['rval']                   # rval = factor
    return loadings.set_index(['Factor', 'Item'])['Est. Std']

rows_2f = {}
for rater_name in ['Self-Report', 'GPT-4', 'GPT-5']:
    rows_2f[rater_name] = get_loadings_2f(rater_name)

compare_2f = pd.DataFrame(rows_2f).round(3)
compare_2f.index.names = ['Factor', 'PHQ-9 Item']
print('2-Factor Model — Standardized Loadings')
display(compare_2f)


2-Factor Model — Standardized Loadings


Self-Report  GPT-4  GPT-5
Factor    PHQ-9 Item                                         
Somatic   Sleep                           0.817  0.839  0.800
          Fatigue                         0.861  0.853  0.847
          Appetite                        0.762  0.735  0.511
          Psychomotor                     0.546  0.776  0.474
Cognitive Anhedonia                       0.841  0.857  0.877
          Depressed Mood                  0.898  0.873  0.805
          Worthlessness / Guilt           0.807  0.761  0.483
          Difficulty Concentrating        0.762  0.778  0.615
          Suicidal Ideation               0.643  0.409  0.341

In [17]:
# Bootstrap CI comparison of factor loadings: GPT-4 vs Self-Report, GPT-5 vs Self-Report
# Uses the same bootstrap indices across all raters per iteration (paired resampling)
# so that within-iteration differences reflect the same individuals.

N_BOOTSTRAP = 500
np.random.seed(42)

data_all = {rater_name: df.loc[common_idx, phq9_items]
            for rater_name, df in raters.items()}
n = len(common_idx)

# boot_diffs[comparison][model][item] = list of (gpt_loading - sr_loading) across bootstrap samples
comparisons = ['GPT-4 vs Self-Report', 'GPT-5 vs Self-Report']
models      = ['1-factor', '2-factor']
boot_diffs  = {cmp: {mdl: {item: [] for item in phq9_items}
                     for mdl in models}
               for cmp in comparisons}

model_descs = {'1-factor': model_1factor, '2-factor': model_2factor}

for b in range(N_BOOTSTRAP):
    if b % 100 == 0:
        print(f'Bootstrap iteration {b}/{N_BOOTSTRAP}...')
    boot_idx = np.random.choice(n, size=n, replace=True)

    # Fit all rater x model combinations on this bootstrap sample
    iter_loads = {}  # (rater, model) -> {item: std_loading}
    for mdl, mdl_desc in model_descs.items():
        for rater_name in ['Self-Report', 'GPT-4', 'GPT-5']:
            try:
                data_boot = data_all[rater_name].iloc[boot_idx]
                m = semopy.Model(mdl_desc)
                m.fit(data_boot)
                insp = m.inspect(std_est=True)
                insp_f = insp[insp['op'] == '~']
                iter_loads[(rater_name, mdl)] = dict(
                    zip(insp_f['lval'], insp_f['Est. Std'].astype(float)))
            except Exception:
                iter_loads[(rater_name, mdl)] = None

    # Accumulate paired differences
    for mdl in models:
        sr   = iter_loads.get(('Self-Report', mdl))
        gpt4 = iter_loads.get(('GPT-4',       mdl))
        gpt5 = iter_loads.get(('GPT-5',       mdl))
        if sr and gpt4:
            for item in phq9_items:
                if item in sr and item in gpt4:
                    boot_diffs['GPT-4 vs Self-Report'][mdl][item].append(gpt4[item] - sr[item])
        if sr and gpt5:
            for item in phq9_items:
                if item in sr and item in gpt5:
                    boot_diffs['GPT-5 vs Self-Report'][mdl][item].append(gpt5[item] - sr[item])

print(f'Done. {N_BOOTSTRAP} bootstrap iterations completed.')


Bootstrap iteration 0/500...
Bootstrap iteration 100/500...
Bootstrap iteration 200/500...
Bootstrap iteration 300/500...
Bootstrap iteration 400/500...
Done. 500 bootstrap iterations completed.


In [18]:
# Multiple-testing correction via Benjamini-Hochberg FDR (family = 9 items per comparison per model)
# Bootstrap p-value: proportion of bootstrap differences on the wrong side of 0, doubled (two-tailed)
# Adjusted p-value: BH-corrected; sig_fdr = True if p_adj < 0.05

def bootstrap_pvalue(diffs):
    """Two-tailed p-value from bootstrap null: p = 2 * min(P(diff>0), P(diff<0))"""
    diffs = np.array(diffs)
    p_pos = np.mean(diffs > 0)
    p_neg = np.mean(diffs < 0)
    return float(2 * min(p_pos, p_neg))

def bh_correct(pvals):
    """Benjamini-Hochberg FDR correction. Returns adjusted p-values."""
    pvals = np.array(pvals)
    k = len(pvals)
    order = np.argsort(pvals)
    ranks = np.empty(k, dtype=int)
    ranks[order] = np.arange(1, k + 1)
    adj = pvals * k / ranks
    # Enforce monotonicity: working backwards
    for i in range(k - 2, -1, -1):
        adj[order[i]] = min(adj[order[i]], adj[order[i + 1]])
    return np.minimum(adj, 1.0)

for cmp in comparisons:
    for mdl in models:
        # Compute raw p-values for all 9 items
        pvals = [bootstrap_pvalue(boot_diffs[cmp][mdl][item]) for item in phq9_items]
        adj_pvals = bh_correct(pvals)

        rows = []
        for item, pval, adj_p in zip(phq9_items, pvals, adj_pvals):
            diffs  = np.array(boot_diffs[cmp][mdl][item])
            mean_d = np.mean(diffs)
            ci_lo  = np.percentile(diffs, 2.5)
            ci_hi  = np.percentile(diffs, 97.5)
            rows.append({
                'PHQ-9 Item': item_labels[item],
                'Mean Diff':  round(mean_d, 3),
                '95% CI':     f'[{ci_lo:.3f}, {ci_hi:.3f}]',
                'p (raw)':    round(pval, 4),
                'p (BH-FDR)': round(adj_p, 4),
                'Sig.':       '*' if adj_p < 0.05 else '',
            })

        result_df = pd.DataFrame(rows).set_index('PHQ-9 Item')
        print(f'\n{cmp} | {mdl} model')
        print('Mean Diff > 0: GPT loading higher; * = BH-FDR adjusted p < 0.05')
        display(result_df)



GPT-4 vs Self-Report | 1-factor model
Mean Diff > 0: GPT loading higher; * = BH-FDR adjusted p < 0.05


,Mean Diff,95% CI,p (raw),p (BH-FDR),Sig.
PHQ-9 Item,,,,,
Anhedonia,0.010,"[-0.028, 0.044]",0.580,0.6525,
Depressed Mood,-0.030,"[-0.056, -0.000]",0.052,0.0936,
Sleep,0.063,"[0.026, 0.099]",0.000,0.0000,*
Fatigue,0.027,"[-0.005, 0.059]",0.088,0.1320,
Appetite,-0.004,"[-0.048, 0.043]",0.884,0.8840,
Worthlessness / Guilt,-0.053,"[-0.091, -0.017]",0.008,0.0180,*
Difficulty Concentrating,0.022,"[-0.014, 0.059]",0.256,0.3291,
Psychomotor,0.192,"[0.147, 0.239]",0.000,0.0000,*
Suicidal Ideation,-0.234,"[-0.291, -0.174]",0.000,0.0000,*



GPT-4 vs Self-Report | 2-factor model
Mean Diff > 0: GPT loading higher; * = BH-FDR adjusted p < 0.05


,Mean Diff,95% CI,p (raw),p (BH-FDR),Sig.
PHQ-9 Item,,,,,
Anhedonia,0.016,"[-0.021, 0.051]",0.340,0.4371,
Depressed Mood,-0.025,"[-0.053, 0.002]",0.072,0.1620,
Sleep,0.021,"[-0.013, 0.058]",0.256,0.3840,
Fatigue,-0.008,"[-0.038, 0.024]",0.600,0.6000,
Appetite,-0.027,"[-0.069, 0.016]",0.244,0.3840,
Worthlessness / Guilt,-0.045,"[-0.083, -0.009]",0.016,0.0480,*
Difficulty Concentrating,0.016,"[-0.025, 0.057]",0.440,0.4950,
Psychomotor,0.229,"[0.184, 0.277]",0.000,0.0000,*
Suicidal Ideation,-0.236,"[-0.293, -0.181]",0.000,0.0000,*



GPT-5 vs Self-Report | 1-factor model
Mean Diff > 0: GPT loading higher; * = BH-FDR adjusted p < 0.05


,Mean Diff,95% CI,p (raw),p (BH-FDR),Sig.
PHQ-9 Item,,,,,
Anhedonia,-0.013,"[-0.057, 0.030]",0.592,0.5920,
Depressed Mood,-0.131,"[-0.176, -0.089]",0.000,0.0000,*
Sleep,-0.032,"[-0.086, 0.022]",0.236,0.2655,
Fatigue,-0.041,"[-0.088, 0.005]",0.080,0.1029,
Appetite,-0.232,"[-0.296, -0.168]",0.000,0.0000,*
Worthlessness / Guilt,-0.345,"[-0.409, -0.286]",0.000,0.0000,*
Difficulty Concentrating,-0.116,"[-0.171, -0.062]",0.000,0.0000,*
Psychomotor,-0.113,"[-0.189, -0.044]",0.000,0.0000,*
Suicidal Ideation,-0.315,"[-0.385, -0.237]",0.000,0.0000,*



GPT-5 vs Self-Report | 2-factor model
Mean Diff > 0: GPT loading higher; * = BH-FDR adjusted p < 0.05


,Mean Diff,95% CI,p (raw),p (BH-FDR),Sig.
PHQ-9 Item,,,,,
Anhedonia,0.036,"[-0.001, 0.074]",0.064,0.0823,
Depressed Mood,-0.093,"[-0.128, -0.061]",0.000,0.0000,*
Sleep,-0.018,"[-0.067, 0.029]",0.436,0.4880,
Fatigue,-0.015,"[-0.055, 0.028]",0.488,0.4880,
Appetite,-0.252,"[-0.318, -0.188]",0.000,0.0000,*
Worthlessness / Guilt,-0.325,"[-0.378, -0.268]",0.000,0.0000,*
Difficulty Concentrating,-0.149,"[-0.204, -0.092]",0.000,0.0000,*
Psychomotor,-0.074,"[-0.155, 0.001]",0.060,0.0823,
Suicidal Ideation,-0.303,"[-0.368, -0.236]",0.000,0.0000,*


In [10]:
# Chi-square difference test: 1-factor is nested within 2-factor
# (2-factor adds 1 free parameter: the inter-factor correlation)
print('=' * 70)
print('Chi-square Difference Test: 1-factor vs 2-factor')
print('=' * 70)

chi2_rows = []
for rater_name in raters:
    s1 = cfa_results[rater_name]['1-factor']['stats']
    s2 = cfa_results[rater_name]['2-factor']['stats']
    chi2_1 = float(s1.loc['Value', 'chi2']);  df_1 = float(s1.loc['Value', 'DoF'])
    chi2_2 = float(s2.loc['Value', 'chi2']);  df_2 = float(s2.loc['Value', 'DoF'])
    delta_chi2 = chi2_1 - chi2_2
    delta_df   = int(df_1 - df_2)
    p_val      = 1 - chi2_dist.cdf(delta_chi2, df=max(delta_df, 1))
    verdict    = '2-factor significantly better' if p_val < 0.05 else 'no significant improvement'
    chi2_rows.append({'Rater': rater_name,
                      'chi2(1-f)': round(chi2_1, 3), 'chi2(2-f)': round(chi2_2, 3),
                      'delta_chi2': round(delta_chi2, 3), 'delta_df': delta_df,
                      'p-value': round(p_val, 4), 'Verdict': verdict})
    print(f'\n{rater_name}: delta_chi2={delta_chi2:.3f}, delta_df={delta_df}, p={p_val:.4f}  ->  {verdict}')

display(pd.DataFrame(chi2_rows).set_index('Rater'))


Chi-square Difference Test: 1-factor vs 2-factor

Self-Report: delta_chi2=136.725, delta_df=1, p=0.0000  ->  2-factor significantly better

GPT-4: delta_chi2=64.305, delta_df=1, p=0.0000  ->  2-factor significantly better

GPT-5: delta_chi2=203.167, delta_df=1, p=0.0000  ->  2-factor significantly better


,chi2(1-f),chi2(2-f),delta_chi2,delta_df,p-value,Verdict
Rater,,,,,,
Self-Report,442.810,306.084,136.725,1,0.0,2-factor significantly better
GPT-4,519.520,455.214,64.305,1,0.0,2-factor significantly better
GPT-5,504.935,301.769,203.167,1,0.0,2-factor significantly better
